In [ ]:
## Just code I started writing and never finished.

def create_anndata_object(directory, sample_id, condition_name):

    adata1 = sc.read_mtx(op.join(directory, f'{sample_id}_matrix.mtx.gz'))
    adata1_bc  = pd.read_csv(op.join(directory, f'{sample_id}_barcodes.tsv.gz'), header = None)
    adata1_f = pd.read_csv(op.join(directory, f'{sample_id}_features.tsv.gz'), header = None)

    adata1  = adata1.T

    adata1_f.columns = ['gene_name']
    adata1_bc.columns = ['cell_id']

    adata1.var['gene_name'] = adata1_f['gene_name'].values
    adata1.obs['cell_id'] = adata1_bc['cell_id'].values

    adata1.obs['condition'] = condition_name
    adata1.obs['barcode'] = adata1.obs['condition']+adata1.obs['cell_id']

    return adata1


In [ ]:
import os.path as op
import os
import scanpy as sc
import pandas as pd
import anndata as ad

# set output directory
output_dir = "/data_nfs/og86asub/netmap/netmap-evaluation/results/cd8_usecase"

# Create both directories regardless of whether they already exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "networks"), exist_ok=True)

directory_name = '/data_nfs/og86asub/netmap/netmap-evaluation/netmap/data/cd8_mouse/72h'
sample_id = 'GSM8286677_10XSC009-02-ATAC'
condition_name = 'exhausted_d3_'

adata_72 = create_anndata_object(directory_name, sample_id, condition_name)

directory_name = '/data_nfs/og86asub/netmap/netmap-evaluation/netmap/data/cd8_mouse/96h'
sample_id = 'GSM8286676_10XSC009-01-ATAC'
condition_name = 'exhausted_d4_'

adata_96 = create_anndata_object(directory_name, sample_id, condition_name)

directory_name = '/data_nfs/og86asub/netmap/netmap-evaluation/netmap/data/cd8_mouse/active'
sample_id = 'GSM8286679_10XSC011-02-ATAC'
condition_name = 'activated_d1_'

adata_active = create_anndata_object(directory_name, sample_id, condition_name)

directory_name = '/data_nfs/og86asub/netmap/netmap-evaluation/netmap/data/cd8_mouse/naive'
sample_id = 'GSM8286678_10XSC011-01-ATAC'
condition_name = 'naive_d4_'

adata_naive = create_anndata_object(directory_name, sample_id, condition_name)




In [ ]:

barcode = pd.read_csv('/data_nfs/og86asub/netmap/netmap-evaluation/netmap/data/cd8_mouse/ga_an0682_10x_gex_exvivo_p14_gp33_stim_int_filtered_cell_barcodes.txt')
barcode.columns = ['barcode']
adata_atac = ad.concat([adata_72, adata_96, adata_active, adata_naive])
adata_atac = adata_atac[adata_atac.obs['barcode'].isin(barcode.barcode)]

In [ ]:
import snapatac2 as snap


In [ ]:
snap.pp.select_features(adata_atac, n_features=250000)
snap.tl.spectral(adata_atac)



In [ ]:

%%time
snap.tl.umap(adata_atac)



In [ ]:
snap.pp.knn(adata_atac)
snap.tl.leiden(adata_atac)

In [ ]:
sc.pl.umap(adata_atac, color=['leiden', 'condition'])

In [ ]:
gene_matrix = snap.pp.make_gene_matrix(adata_atac, snap.genome.hg38)